In [36]:
import torch 
import pandas as pd

from pathlib import Path
from tqdm import tqdm 

from neuralhydrology.datasetzoo import get_dataset
from neuralhydrology.evaluation import get_tester
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.ealstm import EALSTM
from neuralhydrology.utils.config import Config

In [2]:
torch.cuda.get_device_name()

'NVIDIA A40'

In [3]:
# Load the config.
run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525")
cfg = Config(run_dir_path / "config.yml")


In [4]:
# Load the final model configuration.
ea_lstm = EALSTM(cfg=cfg)
final_weigthts = torch.load(str(run_dir_path / "model_epoch030.pt"), map_location="cpu")
ea_lstm.load_state_dict(final_weigthts)

<All keys matched successfully>

In [5]:
# Load dataset (for now just test (later combine all sets)).
scaler = load_scaler(run_dir=run_dir_path)
ds = get_dataset(cfg=cfg, is_train=True, period="validation", scaler=scaler)

100%|██████████| 86/86 [00:05<00:00, 14.68it/s]


In [6]:
#attribute one. 
tester = get_tester(cfg=cfg, run_dir=run_dir_path, period="validation", init_model=True)

In [7]:
raw_results = tester.evaluate(save_results=False, metrics=["NSE"])

# Validation: 100%|██████████| 86/86 [01:48<00:00,  1.26s/it]


Experiment with noise on slope. 

In [11]:
cfg.static_attributes[3]

'slope_mean'

In [39]:
results_df = pd.DataFrame(columns=noise_levels, index=tester.cached_datasets.keys())

In [ ]:

noise_levels = [-1, -0.1, 0.1, 1]

results_df = pd.DataFrame(columns=noise_levels, index=tester.cached_datasets.keys())

for noise in noise_levels: 
    
	tester_copy = tester

	for id in tester_copy.cached_datasets.keys():	

		#print(f"pre slope: {tester_copy.cached_datasets[id]._attributes[id][3]}")
		tester_copy.cached_datasets[id]._attributes[id][3] += noise 
		#print(f"post slope: {tester_copy.cached_datasets[id]._attributes[id][3]}")

	noise_level_results = tester.evaluate(save_results=False, metrics=["NSE"])

	results_df[str(noise)] = [noise_level_results[id]["1D"]["NSE"] for id in noise_level_results.keys()] 

# Validation: 100%|██████████| 86/86 [01:40<00:00,  1.16s/it]


In [ ]:
results_df.renam

Index([-1.0, -0.1, 0.1, 1.0], dtype='float64')

In [67]:
for col in results_df: 
	print(col)
	print(results_df.sort_values(by=col, ascending=True, inplace=True))

-1.0
None
-0.1
None
0.1
None
1.0
None


In [35]:
for id in tester.cached_datasets.keys():
    
	base_nse = raw_results[id]["1D"]["NSE"]
	altered_nse = {}

	for noise in noise_levels: 

		altered_nse[str(noise)] = slope_noise_results[str(noise)][id]["1D"]["NSE"]

	print(f"{id} Base NSE: {base_nse:.2f} | " + ", ".join(f"{level}: {nse:.2f}" for level, nse in altered_nse.items()))


105 Base NSE: 0.80 | -1: 0.65, -0.1: 0.65, 0.1: 0.65, 1: 0.69
106 Base NSE: 0.83 | -1: 0.74, -0.1: 0.74, 0.1: 0.74, 1: 0.76
110 Base NSE: 0.46 | -1: 0.60, -0.1: 0.61, 0.1: 0.60, 1: 0.57
129 Base NSE: 0.53 | -1: -0.42, -0.1: -0.43, 0.1: -0.42, 1: -0.35
136 Base NSE: -0.32 | -1: 0.43, -0.1: 0.43, 0.1: 0.43, 1: 0.37
137 Base NSE: 0.52 | -1: 0.10, -0.1: 0.07, 0.1: 0.10, 1: 0.23
145 Base NSE: 0.53 | -1: 0.50, -0.1: 0.50, 0.1: 0.50, 1: 0.64
15 Base NSE: 0.60 | -1: 0.30, -0.1: 0.29, 0.1: 0.30, 1: 0.33
152 Base NSE: 0.43 | -1: 0.33, -0.1: 0.32, 0.1: 0.33, 1: 0.49
161 Base NSE: -5.23 | -1: -3.30, -0.1: -3.33, 0.1: -3.30, 1: -3.15
175 Base NSE: 0.81 | -1: 0.56, -0.1: 0.55, 0.1: 0.56, 1: 0.60
182 Base NSE: 0.77 | -1: 0.63, -0.1: 0.63, 0.1: 0.63, 1: 0.64
218 Base NSE: 0.39 | -1: 0.42, -0.1: 0.42, 0.1: 0.42, 1: 0.42
221 Base NSE: 0.65 | -1: 0.61, -0.1: 0.61, 0.1: 0.61, 1: 0.69
25 Base NSE: 0.72 | -1: 0.23, -0.1: 0.22, 0.1: 0.23, 1: 0.29
256 Base NSE: 0.61 | -1: 0.68, -0.1: 0.67, 0.1: 0.68, 1: 0.73


In [26]:
slope_noise_results["0.1"]["105"]["1D"]["NSE"]

0.6515105962753296